In [ ]:
"""
==============================================================================
🚀 [Step 4] L-channel CLAHE Preprocessing (Feature Enhancement)
==============================================================================
### @Author       : 한의정 (Data Engineering Lead)
### @Description  : 저대비(Low Contrast) 환경에서 수집된 알약 각인의 식별력 향상을 위한
###                 적응형 히스토그램 평활화(CLAHE) 적용.
### @Logic        : RGB 왜곡 방지를 위해 LAB 색공간 분리 후 L(밝기) 채널만 선명도 보정.
### @Parameter    : clipLimit=2.0, tileGridSize=(8, 8)
### @Input        : letterbox_images/train, letterbox_images/val (in-place 덮어쓰기)
### @Output       : letterbox_images/train, letterbox_images/val (CLAHE 적용)
==============================================================================
"""


In [ ]:
import sys, os

try:
    is_colab = 'google.colab' in str(get_ipython())
except NameError:
    is_colab = False

if is_colab:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass  # 이미 마운트됐거나 VSCode 커널 환경

    REPO_DIR = '/content/pill_detection_project'
    if not os.path.exists(REPO_DIR):
        os.system('git clone https://github.com/wina0901/pill_detection_project.git ' + REPO_DIR)

    # nested 폴더 대응 (pill_detection_project/pill_detection_project/src/ 구조)
    PROJECT_ROOT = REPO_DIR
    nested = os.path.join(REPO_DIR, 'pill_detection_project')
    if os.path.isdir(os.path.join(nested, 'src')):
        PROJECT_ROOT = nested

    sys.path.insert(0, PROJECT_ROOT)
    BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'

else:
    REPO_DIR = os.path.expanduser('~/Desktop/vera_pill')
    sys.path.insert(0, REPO_DIR)
    BASE_DIR = os.path.expanduser('~/Desktop/vera_pill/dataset')

print(f"환경    : {'Colab' if is_colab else '로컬'}")
print(f"REPO_DIR: {sys.path[0]}")
print(f"BASE_DIR: {BASE_DIR}")
assert os.path.exists(BASE_DIR), f"🚨 경로 없음: {BASE_DIR}"


In [ ]:
import os, cv2, glob, random, warnings
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from src.preprocessing.viz_utils import show_samples

plt.rcParams['font.family']        = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')
print("✅ 라이브러리 로드 완료")


## 1. CLAHE 적용 전 샘플 저장 (비교용)

In [ ]:
# ==========================================
# 1. 경로 설정 및 CLAHE 엔진 초기화
# ==========================================
train_dir     = os.path.join(BASE_DIR, 'letterbox_images/train')  # ✅ NB03 출력물
val_dir       = os.path.join(BASE_DIR, 'letterbox_images/val')    # ✅ NB03 출력물
TRAIN_LB_JSON = os.path.join(BASE_DIR, 'train_letterbox.json')
VAL_LB_JSON   = os.path.join(BASE_DIR, 'val_letterbox.json')

assert os.path.exists(train_dir), "NB03을 먼저 실행하세요"

# 💡 [엔지니어링 포인트] CLAHE 엔진 초기화
# - clipLimit (2.0): 대비 제한 임계값. 너무 높으면 먼지(Noise)까지 증폭됩니다.
#   → 2.0이 알약 각인 선명도와 노이즈 억제의 최적 균형점
# - tileGridSize (8, 8): 이미지를 8x8 격자로 쪼개서 '국소적'으로 대비를 조정합니다.
#   → 조명이 불균일한 이미지에서도 각 영역별로 최적화된 대비 조정 가능
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# 비교를 위해 CLAHE 전 이미지 5장을 메모리에 보관
sample_paths = random.sample(glob.glob(os.path.join(train_dir, '*.jpg')), 5)
before_imgs  = []
for p in sample_paths:
    img = cv2.imdecode(np.fromfile(p, np.uint8), cv2.IMREAD_COLOR)
    before_imgs.append((p, cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))

print("✅ CLAHE 전 이미지 5장 저장 완료 (전후 비교용)")


## 2. CLAHE 적용 (in-place)

In [ ]:
def apply_clahe_to_folder(folder_path):
    """
    지정된 폴더의 이미지를 LAB 색공간으로 변환 후,
    밝기(L) 채널의 대비만 극대화하여 덮어씁니다.
    """
    if not os.path.exists(folder_path):
        print(f"🚨 [에러] 폴더 경로를 찾을 수 없습니다: {folder_path}")
        return

    img_list = glob.glob(os.path.join(folder_path, "*.jpg"))
    if not img_list:
        print(f"⚠️  {os.path.basename(folder_path)} 폴더에 처리할 이미지가 없습니다.")
        return

    print(f"🔬 [전처리] {os.path.basename(folder_path)} 폴더 CLAHE 시력 교정 가동 (총 {len(img_list):,}장)")

    for img_path in tqdm(img_list, desc="CLAHE 적용 중"):
        # 1) 디코딩 (한글 경로 및 네트워크 드라이브 I/O 에러 원천 차단)
        img_array = np.fromfile(img_path, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        if img is None: continue

        # 2) BGR → LAB 색공간 변환 (핵심 로직)
        # RGB 채널 전체에 평활화를 걸면 알약 고유의 색상(Hue)이 심하게 왜곡됩니다.
        # LAB 색공간은 밝기(L)와 색상(A, B)을 완전히 분리하므로 왜곡 없는 대비 보정이 가능합니다.
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

        # L: 밝기(Lightness), A/B: 색상 정보 (색상은 건드리지 않음)
        l, a, b = cv2.split(lab)

        # 3) L(밝기) 채널에만 적응형 히스토그램 평활화 적용
        l_clahe = clahe.apply(l)

        # 4) 채널 병합 및 원래 색공간(BGR)으로 복귀
        lab_merged = cv2.merge((l_clahe, a, b))
        final_img  = cv2.cvtColor(lab_merged, cv2.COLOR_LAB2BGR)

        # 5) 디스크 제자리 덮어쓰기 (In-place Update)
        # cv2.imwrite 대신 imencode + tofile 사용 (한글 경로 안전 저장)
        result, encoded_img = cv2.imencode('.jpg', final_img)
        if result:
            with open(img_path, mode='w+b') as f:
                encoded_img.tofile(f)

print("🚀 [Phase 5] 데이터셋 미세 각인 대비(Contrast) 최적화 작업을 시작합니다.\n")

apply_clahe_to_folder(train_dir)
apply_clahe_to_folder(val_dir)

print("\n✨ [데이터 엔지니어링 완수] 모든 이미지의 알약 각인과 음각이 선명하게 복원되었습니다!")
print("   이제 모델이 흐릿한 글자도 놓치지 않고 잡아낼 수 있습니다.")


## 3. 📸 CLAHE 전후 비교 (5장)

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(12, 22))
for i, (p, before_rgb) in enumerate(before_imgs):
    after_img = cv2.imdecode(np.fromfile(p, np.uint8), cv2.IMREAD_COLOR)
    after_rgb = cv2.cvtColor(after_img, cv2.COLOR_BGR2RGB)

    axes[i][0].imshow(before_rgb);  axes[i][0].set_title('CLAHE 전', fontsize=10)
    axes[i][1].imshow(after_rgb);   axes[i][1].set_title('CLAHE 후', fontsize=10)
    for ax in axes[i]:
        ax.axis('off')

plt.suptitle("CLAHE 전후 비교 — 각인 선명도 확인", fontsize=13)
plt.tight_layout(); plt.show()


## 4. 📸 CLAHE 적용 후 Train 20장

In [ ]:
show_samples(train_dir, TRAIN_LB_JSON, n=20, title="CLAHE 적용 후 Train 샘플")


## 5. 📸 CLAHE 적용 후 Val 20장

In [ ]:
show_samples(val_dir, VAL_LB_JSON, n=20, title="CLAHE 적용 후 Val 샘플")


## 6. 픽셀 통계 확인

In [ ]:
imgs_all = glob.glob(os.path.join(train_dir, '*.jpg'))
sample50 = random.sample(imgs_all, min(50, len(imgs_all)))

means, stds = [], []
for p in sample50:
    img = cv2.imdecode(np.fromfile(p, np.uint8), cv2.IMREAD_COLOR).astype(np.float32) / 255.
    means.append(img.mean(axis=(0,1)))
    stds.append(img.std(axis=(0,1)))

means = np.array(means).mean(axis=0)
stds  = np.array(stds).mean(axis=0)
print(f"픽셀 평균 (BGR): {means.round(3)}")
print(f"픽셀 표준편차   : {stds.round(3)}")
print("\n✅ CLAHE 완료 — NB05(DataLoader)로 진행하세요")


## ✅ NB04 완료
> 다음 단계: NB05 - PyTorch DataLoader 검증